In [1]:
!pip install pandas scikit-learn

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = """product,color,category,style,occasion
Black Shirt,Black,Top,Casual,Casual
White Shirt,White,Top,Casual,Casual
Blue Shirt,Blue,Top,Casual,Casual
Red Shirt,Red,Top,Casual,Casual
Green Shirt,Green,Top,Casual,Casual
Formal White Shirt,White,Top,Formal,Office
Formal Blue Shirt,Blue,Top,Formal,Office
Formal Black Shirt,Black,Top,Formal,Office
Polo T-Shirt,Navy,Top,Sports,Casual
Graphic T-Shirt,Black,Top,Streetwear,Casual
Blue Jeans,Blue,Bottom,Casual,Casual
Black Jeans,Black,Bottom,Casual,Casual
Grey Jeans,Grey,Bottom,Casual,Casual
Khaki Chinos,Khaki,Bottom,SmartCasual,Office
Black Trouser,Black,Bottom,Formal,Office
Grey Trouser,Grey,Bottom,Formal,Office
Navy Trouser,Navy,Bottom,Formal,Office
Cargo Pants,Olive,Bottom,Streetwear,Casual
Joggers,Black,Bottom,Sports,Casual
Track Pants,Grey,Bottom,Sports,Casual
White Sneakers,White,Footwear,Casual,Casual
Black Sneakers,Black,Footwear,Casual,Casual
Running Shoes,Blue,Footwear,Sports,Casual
Basketball Shoes,Red,Footwear,Sports,Casual
Leather Shoes,Brown,Footwear,Formal,Office
Oxford Shoes,Black,Footwear,Formal,Office
Loafers,Tan,Footwear,SmartCasual,Office
Sandals,Brown,Footwear,Casual,Beach
Flip Flops,Blue,Footwear,Casual,Beach
Boots,Brown,Footwear,Winter,Casual
Silver Watch,Silver,Accessory,Casual,Casual
Black Watch,Black,Accessory,Formal,Office
Leather Belt,Brown,Accessory,Formal,Office
Black Belt,Black,Accessory,Formal,Office
Sunglasses,Black,Accessory,Casual,Beach
Cap,Navy,Accessory,Sports,Casual
Backpack,Black,Accessory,Casual,Travel
Messenger Bag,Brown,Accessory,SmartCasual,Office
Tie,Red,Accessory,Formal,Office
Pocket Square,White,Accessory,Formal,Office
Navy Blazer,Navy,Outerwear,Formal,Office
Black Blazer,Black,Outerwear,Formal,Office
Denim Jacket,Blue,Outerwear,Casual,Casual
Leather Jacket,Black,Outerwear,Biker,Casual
Hoodie,Grey,Outerwear,Streetwear,Casual
Sweatshirt,Maroon,Outerwear,Casual,Casual
Winter Coat,Black,Outerwear,Winter,Casual
Party Dress,Red,Dress,Party,Party
Cocktail Dress,Black,Dress,Party,Party
Summer Dress,Yellow,Dress,Casual,Vacation"""

with open('fashion_products.csv', 'w') as f:
    f.write(data)

df = pd.read_csv('fashion_products.csv')
df.head()

,product,color,category,style,occasion
0,Black Shirt,Black,Top,Casual,Casual
1,White Shirt,White,Top,Casual,Casual
2,Blue Shirt,Blue,Top,Casual,Casual
3,Red Shirt,Red,Top,Casual,Casual
4,Green Shirt,Green,Top,Casual,Casual


In [4]:
df["combined"] = (
    df["color"] + " " +
    df["category"] + " " +
    df["style"] + " " +
    df["occasion"]
)

df.head()

,product,color,category,style,occasion,combined
0,Black Shirt,Black,Top,Casual,Casual,Black Top Casual Casual
1,White Shirt,White,Top,Casual,Casual,White Top Casual Casual
2,Blue Shirt,Blue,Top,Casual,Casual,Blue Top Casual Casual
3,Red Shirt,Red,Top,Casual,Casual,Red Top Casual Casual
4,Green Shirt,Green,Top,Casual,Casual,Green Top Casual Casual


In [5]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    df["combined"]
)

print(tfidf_matrix.shape)

(50, 32)


In [8]:
similarity = cosine_similarity(tfidf_matrix)

print(similarity.shape)

(50, 50)


In [6]:
def recommend(product_name):

    if product_name not in df["product"].values:
        return "Product not found"

    idx = df[df["product"] == product_name].index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i in scores[1:6]:
        recommendations.append(
            df.iloc[i[0]]["product"]
        )

    return recommendations

In [9]:
recommend("Black Shirt")

['Blue Shirt',
 'Graphic T-Shirt',
 'Black Jeans',
 'Black Sneakers',
 'White Shirt']

In [10]:
def recommend_outfit(product):

    product_row = df[df["product"] == product]

    if len(product_row) == 0:
        return "Product not found"

    occasion = product_row.iloc[0]["occasion"]

    bottoms = df[
        (df["category"] == "Bottom") &
        (df["occasion"] == occasion)
    ]

    footwear = df[
        (df["category"] == "Footwear") &
        (df["occasion"] == occasion)
    ]

    accessories = df[
        (df["category"] == "Accessory")
    ]

    return {
        "Selected Product": product,
        "Bottom": bottoms.iloc[0]["product"],
        "Footwear": footwear.iloc[0]["product"],
        "Accessory": accessories.iloc[0]["product"]
    }

In [11]:
recommend_outfit("Black Shirt")

{'Selected Product': 'Black Shirt',
 'Bottom': 'Blue Jeans',
 'Footwear': 'White Sneakers',
 'Accessory': 'Silver Watch'}

In [12]:
import pickle

pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))

print("Files saved successfully!")

Files saved successfully!


In [25]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [20]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "Outfit Recommendation API Running"}

In [21]:
@app.get("/recommend")
def get_recommendation(product: str):

    result = recommend(product)

    return {
        "product": product,
        "recommendations": result
    }

In [22]:
@app.get("/outfit")
def get_outfit(product: str):

    result = recommend_outfit(product)

    return result

In [23]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

In [28]:
from pyngrok import ngrok

ngrok.set_auth_token("3EwPmBujhDxMkWOIYpiCYNZRzpP_2TnF4pjcm4BtG6Erh7QBU")

In [29]:
public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://strainer-playback-hunger.ngrok-free.dev" -> "http://localhost:8000"


In [32]:
from threading import Thread
import uvicorn

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server = Thread(target=run_api)
server.start()